In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

models = {
    "G5": "C:/GitHub/NLP_Project/models/xlm-roberta-clickbait",
    "G7": "C:/GitHub/NLP_Project/models/xlm-roberta-clickbait-g7",
    "G8": "C:/GitHub/NLP_Project/models/xlm-roberta-clickbait-g8",
}

loaded = {}
for name, path in models.items():
    tok = AutoTokenizer.from_pretrained(path)
    mdl = AutoModelForSequenceClassification.from_pretrained(path)
    mdl.eval()
    loaded[name] = (tok, mdl)

def predict(name, title, content=""):
    tok, mdl = loaded[name]
    inputs = tok(title, content, return_tensors="pt", truncation=True, max_length=256)
    with torch.no_grad():
        logits = mdl(**inputs).logits
    return torch.softmax(logits, dim=-1)[0][1].item()

def run_cases(cases, expected_label, section):
    """cases: list of (title, note). expected_label: 0=non-CB, 1=CB"""
    print(f"\n{'='*70}")
    print(f"【{section}】  expected label={expected_label}  ({'應判 0，FP 為誤判' if expected_label==0 else '應判 1，FN 為漏判'})")
    print(f"{'='*70}")
    for title, note in cases:
        row = f"  {note[:45]:<45}"
        for name in ["G5", "G7", "G8"]:
            p = predict(name, title)
            wrong = (p > 0.5) != bool(expected_label)
            row += f"  {name} {'❌' if wrong else '✓'} {p:.3f}"
        print(row)

# ──────────────────────────────────────────────────────────────────────
# 區塊 1：Bug 3 — 中文驚訝類轉折詞（label=0，應全判 0）
# 目標：G8 tw_neg/tw_pos 對「竟/居然/沒想到」的修復是否有效
# ──────────────────────────────────────────────────────────────────────
zh_bug3 = [
    ("台灣男子騎腳踏車環島，竟只花了8天完成全程",         "竟 + 具體天數（完整）"),
    ("研究發現每天走路30分鐘，竟可降低心臟病風險40%",     "竟 + 具體數字（完整）"),
    ("八旬老翁參加馬拉松，竟以4小時完賽超越眾多年輕人",   "竟 + 具體時間（完整）"),
    ("颱風過後竟在淡水河發現活生生的鱷魚",               "竟 + 荒誕但完整"),
    ("科學家居然在深海發現會發光的新品種水母",           "居然 + 完整資訊"),
    ("台北廢棄大樓居然住了超過200隻流浪貓",             "居然 + 具體數字（完整）"),
    ("沒想到每天喝綠茶真的能改善睡眠，研究追蹤1000人確認", "沒想到 + 具體研究（完整）"),
    ("沒想到這次投票率高達75%，創下近20年新高",         "沒想到 + 具體數字（完整）"),
    ("不料強震發生後72小時，搜救隊仍在廢墟中救出生還者", "不料 + 完整事件"),
    ("萬萬沒想到，這款平價護膚品的成分與百元保養品完全相同", "萬萬沒想到 + 完整比較"),
]
run_cases(zh_bug3, expected_label=0, section="Bug 3：中文驚訝類轉折詞 FP 測試（label=0）")

# ──────────────────────────────────────────────────────────────────────
# 區塊 2：ETtoday OOD 退步診斷（label=0 和 label=1 各幾筆）
# G5=1.000 → G8=0.444，要看是 FP 多還是 FN 多
# ──────────────────────────────────────────────────────────────────────
ettoday_neg = [
    ("林口長庚急診壅塞 醫師籲輕症患者勿湧入", "硬新聞平實標題 label=0"),
    ("台中市議會今通過109年度總預算 規模創歷年新高", "政治硬新聞 label=0"),
    ("北市府：信義計畫區停車費率將依時段調整", "政策公告 label=0"),
    ("氣象局：今晚鋒面南下 北台灣明顯降溫", "天氣新聞 label=0"),
    ("勞動部統計：去年製造業加班費年增3.2%", "統計數字 label=0"),
]
ettoday_pos = [
    ("竟然是這個原因！北市信義區停電背後的真相曝光", "竟 + 懸念 label=1"),
    ("他只吃這一樣東西，30天後身體出現驚人變化", "清單誘餌 label=1"),
    ("醫師不說的秘密：某類早餐長期吃恐傷腎，你還在吃嗎？", "知性語氣懸念 label=1"),
    ("台灣最美小鎮榜單出爐！第一名連在地人都沒去過", "清單懸念 label=1"),
    ("財經達人曝光：存到第一桶金的人都有這個習慣", "財經懸念 label=1"),
]
run_cases(ettoday_neg, expected_label=0, section="ETtoday OOD 診斷：平實標題 FP 測試（label=0）")
run_cases(ettoday_pos, expected_label=1, section="ETtoday OOD 診斷：clickbait FN 測試（label=1）")

# ──────────────────────────────────────────────────────────────────────
# 區塊 3：Bug 5 — 英文知性語氣（label=1，確認是否系統性漏判）
# rt_en_b4_invest 三個模型全滅，這裡加 4 筆確認不是特例
# ──────────────────────────────────────────────────────────────────────
en_bug5 = [
    ("Assessing the Long-term Financial Sustainability of Household Portfolios: Certain Low-risk Investment Vehicles Offer Hidden Disadvantages Over Time",
     "投資知性語氣（rt_en_b4_invest 原案例）"),
    ("Examining the Neurological Basis of Chronic Fatigue: A Specific Category of Dietary Pattern May Accelerate Cognitive Decline",
     "健康學術語氣 + 模糊代稱"),
    ("Researchers Explore the Socioeconomic Factors Behind Retirement Savings Gaps: One Common Habit Among Middle-income Earners Proves Surprisingly Counterproductive",
     "學術語氣 + surprisingly + 懸念"),
    ("A Comprehensive Review of Sleep Architecture and Its Disruption: Certain Evening Behaviors Widely Considered Harmless Are Linked to Reduced REM Duration",
     "睡眠研究知性語氣 + 懸念"),
]
run_cases(en_bug5, expected_label=1, section="Bug 5：英文知性語氣 FN 測試（label=1）")

# ──────────────────────────────────────────────────────────────────────
# 區塊 4：英文驚訝類詞（原有測試，保留作對照）
# ──────────────────────────────────────────────────────────────────────
en_surprise = [
    ("Researchers found that daily coffee drinkers surprisingly have lower risk of type 2 diabetes", "surprisingly + 具體結論"),
    ("A 90-year-old man unexpectedly finished the Boston Marathon in under 5 hours",                 "unexpectedly + 具體時間"),
    ("Scientists surprisingly discovered a new species of frog in downtown Tokyo",                   "surprisingly + 完整發現"),
    ("Turns out walking barefoot on grass actually reduces stress, study of 1,000 people confirms",  "turns out + 具體研究"),
    ("Shockingly, voter turnout in this year's election hit 80%, the highest in 30 years",           "shockingly + 具體數字"),
    ("You won't believe it, but the city's air quality improved 40% after the new policy",           "you won't believe + 具體數字"),
    ("Little did they know, the abandoned building had been home to over 300 stray cats for years",  "little did they know（懸念殘留）"),
    ("To everyone's surprise, the underdog team won the championship by 20 points",                  "to everyone's surprise + 具體分數"),
]
run_cases(en_surprise, expected_label=0, section="英文驚訝類詞 FP 測試（label=0，對照組）")



【Bug 3：中文驚訝類轉折詞 FP 測試（label=0）】  expected label=0  (應判 0，FP 為誤判)
  竟 + 具體天數（完整）                                   G5 ❌ 0.998  G7 ❌ 0.987  G8 ❌ 0.986
  竟 + 具體數字（完整）                                   G5 ❌ 0.999  G7 ❌ 0.998  G8 ❌ 0.996
  竟 + 具體時間（完整）                                   G5 ❌ 0.999  G7 ❌ 0.998  G8 ❌ 0.996
  竟 + 荒誕但完整                                      G5 ❌ 0.659  G7 ❌ 0.972  G8 ✓ 0.122
  居然 + 完整資訊                                      G5 ❌ 0.991  G7 ❌ 0.989  G8 ❌ 0.934
  居然 + 具體數字（完整）                                  G5 ❌ 0.995  G7 ❌ 0.995  G8 ❌ 0.992
  沒想到 + 具體研究（完整）                                 G5 ❌ 0.999  G7 ❌ 0.995  G8 ❌ 0.991
  沒想到 + 具體數字（完整）                                 G5 ❌ 0.992  G7 ❌ 0.987  G8 ❌ 0.996
  不料 + 完整事件                                      G5 ✓ 0.006  G7 ✓ 0.019  G8 ✓ 0.037
  萬萬沒想到 + 完整比較                                   G5 ❌ 1.000  G7 ❌ 0.999  G8 ❌ 0.995

【ETtoday OOD 診斷：平實標題 FP 測試（label=0）】  expected label=0  (應判 0，FP 為誤判)
  硬新聞平實標題 label=0      